# NVIDIA NeMo Agent Toolkit — Basics

[NeMo Agent Toolkit](https://docs.nvidia.com/nemo/agent-toolkit/latest/index.html) (package name `nvidia-nat`, previously called *AIQToolkit*) is a
framework-agnostic Python library for building, running, and improving AI agent workflows. This notebook covers the
core building blocks you need before doing anything more advanced (like the prompt-optimization notebook that follows
this one).

**Core concepts**

| Concept | What it is |
|---|---|
| **Workflow** | A YAML (or JSON) file describing the `functions`, `llms`, and `workflow` that make up an agent |
| **Function / Tool** | A discrete capability an agent can call (e.g. `current_datetime`, a web search, a calculator) |
| **LLM** | A named reference to a model provider (e.g. an NVIDIA NIM endpoint) used by one or more functions |
| **Workflow entry point** | The top-level agent (e.g. `react_agent`) that decides which tools to call and produces the final answer |

**What we'll do**
1. Install the toolkit and set an API key
2. Explore the built-in components with the `nat` CLI
3. Write a minimal workflow config and run it with `nat run`
4. Run the same workflow from Python instead of the CLI
5. Build and reuse a workflow object programmatically with `WorkflowBuilder`

Reference: [NeMo Agent Toolkit docs](https://docs.nvidia.com/nemo/agent-toolkit/latest/index.html) · [GitHub repo](https://github.com/NVIDIA/NeMo-Agent-Toolkit)

## 0. Setup

Requires Python 3.11, 3.12, or 3.13. If you created your environment from `requirements.txt` in this repo (via `uv`),
the toolkit is already installed and you can skip the `pip install` cell below.

In [ ]:
# Only needed if you didn't install requirements.txt into this kernel's environment already.
# % pip install -q "nvidia-nat[langchain]==1.8.0"

### API key

The examples below use NVIDIA's hosted NIM microservices, which need an `NVIDIA_API_KEY`. Get a free key by signing
in at [build.nvidia.com](https://build.nvidia.com/), opening any model, and generating a personal API key.

In [ ]:
import getpass
import os

if not os.environ.get("NVIDIA_API_KEY"):
    os.environ["NVIDIA_API_KEY"] = getpass.getpass("Enter your NVIDIA_API_KEY (from build.nvidia.com): ")
else:
    print("NVIDIA_API_KEY already set.")

In [ ]:
!nat --version

## 1. Explore installed components

The `nat info components` command lists every component (function/tool, LLM provider, embedder, etc.) that is
registered in your current environment. This is the fastest way to discover what's available without reading
source code.

In [ ]:
!nat info components -t function

## 2. Write a minimal workflow config

Every workflow config has (up to) three top-level sections:

- `functions:` — the tools available to the agent, keyed by the name you'll reference them by. `_type` selects
  which registered component implements it.
- `llms:` — named references to model providers. `_type: nim` uses an NVIDIA NIM endpoint. `base_url` is driven by
  an optional `NVIDIA_BASE_URL` environment variable — see the README for details.
- `workflow:` — the entry point agent. `_type: react_agent` is a [ReAct](https://arxiv.org/abs/2210.03629) agent that
  interleaves reasoning ("Thought") with tool calls ("Action") until it produces a "Final Answer".

`current_datetime` is a built-in tool that ships with the core package (no extra services or API keys required),
so it's a safe, dependency-free way to see the whole pipeline work end to end.

In [ ]:
%%writefile workflow.yml
functions:
  current_datetime:
    _type: current_datetime

llms:
  nim_llm:
    _type: nim
    model_name: nvidia/nvidia/Nemotron-3-Nano-30B-A3B
    temperature: 0.0
    api_key: ${NVIDIA_API_KEY}
    base_url: ${NVIDIA_BASE_URL}
    chat_template_kwargs:
      enable_thinking: false

workflow:
  _type: react_agent
  tool_names: [current_datetime]
  llm_name: nim_llm
  verbose: true
  parse_agent_response_max_retries: 3

## 3. Run it with the `nat` CLI

`nat run` loads a config file, runs the workflow once with the given `--input`, and prints the result. Because
`verbose: true` is set, you'll also see the agent's intermediate `Thought` / `Action` / `Observation` steps.

In [ ]:
!nat run --config_file workflow.yml --input "What time is it right now, and in one sentence, what does a ReAct agent do?"

Other useful CLI entry points built on the same config file:

- `nat serve --config_file workflow.yml` — expose the workflow as a web service (`/generate` endpoint)
- `nat mcp serve --config_file workflow.yml` — expose the workflow as an [MCP](https://modelcontextprotocol.io/) server
- `nat eval --config_file workflow.yml` — score the workflow against a dataset (see notebook 2)

## 4. Run it from Python instead of the CLI

For embedding a workflow inside a larger Python program (or a notebook), use `nat.utils.run_workflow`. It loads a
config file (or an already-loaded `Config` object) and runs it once, just like `nat run` does internally.

In [ ]:
from nat.utils import run_workflow

# Jupyter supports top-level `await`, so we don't need asyncio.run() here.
result = await run_workflow(config_file="workflow.yml", prompt="What day of the week is it today?")
print(result)

## 5. Build a reusable workflow object

`run_workflow` rebuilds the entire workflow (LLM clients, tools, etc.) on every call, which is wasteful if you're
asking many questions in a loop. `WorkflowBuilder` lets you build the workflow once and reuse it, giving you direct
access to the `Runner` object (which also supports streaming via `runner.result_stream()`).

In [ ]:
from nat.builder.workflow_builder import WorkflowBuilder
from nat.runtime.loader import load_config

config = load_config("workflow.yml")

questions = [
    "What time is it right now?",
    "In one short sentence, what is a NeMo Agent Toolkit 'function'?",
]

async with WorkflowBuilder.from_config(config) as builder:
    workflow = await builder.build()
    for question in questions:
        async with workflow.run(question) as runner:
            answer = await runner.result(to_type=str)
            print(f"Q: {question}\nA: {answer}\n")

## 6. Adding more tools

Extending an agent is usually just a matter of adding another entry to `functions:` and listing its name in
`workflow.tool_names`. The toolkit ships several built-in tools you can drop in the same way we used
`current_datetime` above, for example:

- `webpage_query` — RAG over a specific web page (needs an `embedders:` entry, e.g. `nv-embedqa-e5-v5`)
- `code_execution` — runs LLM-generated Python in a sandbox (needs a sandbox service)
- `exa_internet_search` (from the `langchain` extra) — general web search (needs an `EXA_API_KEY`)

See [Add Tools to a Workflow](https://github.com/NVIDIA/NeMo-Agent-Toolkit/blob/main/docs/source/get-started/tutorials/add-tools-to-a-workflow.md)
for worked examples of each, and [Create a New Tool and Workflow](https://github.com/NVIDIA/NeMo-Agent-Toolkit/blob/main/docs/source/get-started/tutorials/create-a-new-workflow.md)
for writing your own custom tool with `@register_function`.

## Next steps

Continue to **`02_prompt_optimization.ipynb`** to see how the toolkit's built-in optimizer can automatically rewrite
an agent's prompt (and tune its hyperparameters) to improve measured accuracy on a dataset.